# Compare Test Runs

Set `MODEL_WEIGHTS` to the checkpoints you want to compare, then run all cells.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

import src

PROJECT_ROOT = Path(src.__file__).resolve().parents[1]
PYTHON_BIN = sys.executable

TEST_MANIFEST = PROJECT_ROOT / "data" / "test" / "a-touch-of-zen.csv"
MODEL_WEIGHTS = [
    PROJECT_ROOT / "src" / "models" / "saved_weights" / "MobileNetV3_v1" / "best_val.pt",
    # PROJECT_ROOT / "src" / "models" / "saved_weights" / "CNN_v1" / "best_val.pt",
]

AUTO_THRESHOLD = True
THRESHOLD = 0.50
RESULTS_LOG = PROJECT_ROOT / "src" / "models" / "test_results.csv"

COMPARE_TAG = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
COMPARE_OUTPUT_ROOT = PROJECT_ROOT / "src" / "models" / "test_comparisons" / COMPARE_TAG
COMPARE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Python: {PYTHON_BIN}")
print(f"Manifest: {TEST_MANIFEST}")
print(f"Compare output dir: {COMPARE_OUTPUT_ROOT}")


Python: /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/.venv/bin/python
Manifest: /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/data/test/a-touch-of-zen.csv
Compare output dir: /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/src/models/test_comparisons/20260306_005648


In [2]:
if not TEST_MANIFEST.exists():
    raise FileNotFoundError(f"Test manifest not found: {TEST_MANIFEST}")
if not MODEL_WEIGHTS:
    raise ValueError("MODEL_WEIGHTS is empty. Add at least one checkpoint path.")

run_summaries = []

for checkpoint in MODEL_WEIGHTS:
    ckpt_path = Path(checkpoint).expanduser().resolve()
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    run_name = ckpt_path.parent.name
    output_dir = COMPARE_OUTPUT_ROOT / run_name

    cmd = [
        PYTHON_BIN,
        "-m",
        "src.test.test",
        "--checkpoint",
        str(ckpt_path),
        "--test_manifest",
        str(TEST_MANIFEST),
        "--output_dir",
        str(output_dir),
        "--results_log",
        str(RESULTS_LOG),
    ]

    if AUTO_THRESHOLD:
        cmd.append("--auto_threshold")
    else:
        cmd.extend(["--threshold", str(THRESHOLD)])

    print("\nRunning:")
    print(" ".join(cmd))
    completed = subprocess.run(cmd, check=False, capture_output=True, text=True)

    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(f"Test command failed for {ckpt_path}")

    summary_path = output_dir / "summary.json"
    if not summary_path.exists():
        raise FileNotFoundError(f"Expected summary not found: {summary_path}")

    summary = json.loads(summary_path.read_text())
    summary["output_dir"] = str(output_dir)

    test_manifest_path = Path(summary.get("test_manifest", TEST_MANIFEST))
    summary["dataset"] = test_manifest_path.stem

    ckpt_data = torch.load(ckpt_path, map_location="cpu")
    trained_labels = [str(x).strip().lower() for x in ckpt_data.get("classes", []) if str(x).strip()]
    summary["num_trained_labels"] = len(trained_labels)
    summary["trained_labels"] = ", ".join(trained_labels)

    run_summaries.append(summary)

print(f"\nCompleted {len(run_summaries)} evaluations.")



Running:
/Users/hughsignoriello/Developer/ml-based-analysis-of-sound/.venv/bin/python -m src.test.test --checkpoint /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/src/models/saved_weights/MobileNetV3_v1/best_val.pt --test_manifest /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/data/test/a-touch-of-zen.csv --output_dir /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/src/models/test_comparisons/20260306_005648/MobileNetV3_v1 --results_log /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/src/models/test_results.csv --auto_threshold

Completed 1 evaluations.


In [3]:
if not run_summaries:
    raise ValueError("No results to compare.")

comparison_df = pd.DataFrame(run_summaries)
ordered_cols = [
    "run_name",
    "dataset",
    "num_trained_labels",
    "trained_labels",
    "checkpoint_path",
    "test_manifest",
    "threshold",
    "micro_f1",
    "macro_f1",
    "subset_accuracy",
    "hamming_accuracy",
    "micro_precision",
    "micro_recall",
    "num_samples",
    "output_dir",
]
ordered_cols = [c for c in ordered_cols if c in comparison_df.columns]
comparison_df = comparison_df[ordered_cols].sort_values("micro_f1", ascending=False).reset_index(drop=True)

display(comparison_df)


,run_name,checkpoint_path,test_manifest,threshold,micro_f1,macro_f1,subset_accuracy,hamming_accuracy,micro_precision,micro_recall,num_samples,output_dir
0,MobileNetV3_v1,/Users/hughsignoriello/Developer/ml-based-anal...,/Users/hughsignoriello/Developer/ml-based-anal...,0.05,0.16092,0.102757,0.082353,0.77098,0.4375,0.098592,85,/Users/hughsignoriello/Developer/ml-based-anal...


In [4]:
comparison_csv = COMPARE_OUTPUT_ROOT / "comparison.csv"
comparison_df.to_csv(comparison_csv, index=False)
print(f"Saved comparison table: {comparison_csv}")

Saved comparison table: /Users/hughsignoriello/Developer/ml-based-analysis-of-sound/src/models/test_comparisons/20260306_005648/comparison.csv
